In [1]:
import os

from google.cloud import vision
from google.cloud.vision_v1 import types

import pandas as pd
import cv2
import numpy as np
import json

import imutils
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

import random
import logging

from Read import Download,Crop,PosiLetter, Vision, AddData, GetPosi, ExtractData, Show, Organize

#Load index list
Year='1941'
Showa='16'


In [2]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [6]:
#Load Data Frame
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+str(Year)+"\\DataFrame_PositionFin.json"
with open(path, 'r') as j:
     dta = json.loads(j.read())

df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+str(Showa)+'_v2.csv')
df=df.drop(df.columns[0], axis=1)

filepath="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year

# 1. Sample Code with single unit

In [70]:
n=44
print(n)
Row=df.iloc[n]
Dept,Office=Row['Dept'],Row['Office']
Posi=GetPosi(Year,dta,Dept,Office)
Dept,Office,Posi

44


('厚生局', '軍事援護課', 'Clerk1')

In [71]:
dta[Year][Dept][Office]

{'StrPage': 40,
 'StrLocation': 320,
 'StrPart': 'Btm',
 'EndPage': 41,
 'EndPart': 'Top',
 'EndLocation': 230,
 'Page_Range': [40, 41],
 'Positions': {'Manager': {'StrPage': 1.0,
   'StrLocation': 303.0,
   'Part': '1',
   'EndLocation': 150.0,
   'EndPage': 1.0},
  'Clerk1': {'StrPage': 1.0,
   'StrLocation': 150.0,
   'Part': '1',
   'EndLocation': 0.0,
   'EndPage': 41.0}}}

In [72]:
MainDF=pd.DataFrame(columns=['Dept', 'Office','Posi','Page','StartPosi','FemaleCount','Size'])
Part=int(dta[Year][Dept][Office]['Positions'][Posi]['Part'])
StrPage=int(dta[Year][Dept][Office]['Positions'][Posi]['StrPage'])
EndPage=int(dta[Year][Dept][Office]['Positions'][Posi]['EndPage'])
PageRange=range(1,StrPage-EndPage+2)
try:    
    for Page in PageRange:
        df1={'Dept':'', 'Office':'','Posi':'','Page':''}
        df1['Dept'],df1['Office'],df1['Posi'],df1['Page']=Dept,Office,Posi,Page
        df1=pd.DataFrame([df1])
        
        # Download Image
        image=Download(filepath,Dept,Office,Posi,Page)
        # Separates Image into Parts
        ImgList=Crop(image,client,Posi,Part,Page)
        StartPosi=ImgList[3]
        Show(ImgList)
        #Extract Data
        DictData=ExtractData(ImgList,client)
        #Aggregate data
        Data=Organize(DictData)
        df2 = pd.DataFrame([Data])
        
        df = pd.concat([df1, df2],axis=1)
        MainDF=pd.concat([MainDF,df], ignore_index = True)
        MainDF.reset_index()
except FileNotFoundError as e:
    print(f"File not found!")

In [74]:
Page=1
image=Download(filepath,Dept,Office,Posi,2)
ImgList=Crop(image,client,Posi,Part,Page)
StartPosi=ImgList[3]

Show(ImgList)

TypeError: 'NoneType' object is not subscriptable

In [78]:
cv2.imshow('',image)
cv2.waitKey(0)

-1

In [8]:
IndexList=[80,171,21,2,92,87,152,87]
np.sort(IndexList)

array([  2,  21,  80,  87,  87,  92, 152, 171])

# 2. Main Code

In [80]:
MainDF=pd.DataFrame(columns=['Dept', 'Office','Posi','Page','StartPosi','FemaleCount','Size'])
NoImageOffice=pd.DataFrame(columns=['Dept', 'Office','Posi','Page'])
IndexError=pd.DataFrame(columns=['Dept', 'Office','Posi','Page'])

for n in random.sample(range(0,len(df)),50):
    Row=df.iloc[n]
    Dept,Office=Row['Dept'],Row['Office']
    display(n,Dept,Office)
    try:
        Posi=GetPosi(Year,dta,Dept,Office)
    except:
        continue
    if Posi=='NA':
        dict = {'Dept':[Dept],
            'Office':[Office],
            'Posi':['NoOffice'],
            'Page':['NoOffice'],
            'StartPosi':['NoOffice'],
            'FemaleCount':['NoOffice'],
            'Size':['NoOffice']
            }
        df1 = pd.DataFrame(dict)        
        MainDF = pd.concat([MainDF, df1], ignore_index = True)
        
    if Posi!='Worker':
        continue

    Part=int(dta[Year][Dept][Office]['Positions'][Posi]['Part'])
    StrPage=int(dta[Year][Dept][Office]['Positions'][Posi]['StrPage'])
    EndPage=int(dta[Year][Dept][Office]['Positions'][Posi]['EndPage'])
    PageRange=range(1,StrPage-EndPage+2)
    try:    
        for Page in PageRange:
            df1={'Dept':'', 'Office':'','Posi':'','Page':''}
            df1['Dept'],df1['Office'],df1['Posi'],df1['Page']=Dept,Office,Posi,Page
            df1=pd.DataFrame([df1])

            # Download Image
            image=Download(filepath,Dept,Office,Posi,Page)
            # Separates Image into Parts
            ImgList=Crop(image,client,Posi,Part,Page)
            StartPosi=ImgList[3]
            
            #Extract Data
            DictData=ExtractData(ImgList,client)
            #Aggregate data
            Data=Organize(DictData)
            df2 = pd.DataFrame([Data])

            NewDF = pd.concat([df1, df2],axis=1)
            print(n)
            display(NewDF)
            MainDF=pd.concat([MainDF,NewDF], ignore_index = True)
            MainDF.reset_index()
    except FileNotFoundError as e:
        data={'Dept':[Dept],'Office':[Office],'Posi':[Posi],'Page':[Page]}
        df3=pd.DataFrame.from_dict(data)
        NoImageOffice=pd.concat([NoImageOffice,df3])
        continue
    except:
        data={'Dept':[Dept],'Office':[Office],'Posi':[Posi],'Page':[Page]}
        df3=pd.DataFrame.from_dict(data)
        IndexError=pd.concat([NoImageOffice,df3])
        continue

76

'厚生局'

'病院'

52

'厚生局'

...

'土木局'

'道路建設課'

95

'経済局'

'農漁課'

In [79]:
dta

{'1941': {'Admin': {'秘書課': {'Starting_Page': 3,
    'StrLocation': 119,
    'StrPart': 'Top',
    'EndPage': 4,
    'EndPart': 'Top',
    'EndLocation': 356,
    'Page_Range': [3, 4]}},
  '中央卸売市場': {'管理課': {'StrPage': 86,
    'StrLocation': 355,
    'StrPart': 'Top',
    'EndPage': 86,
    'EndLocation': 255,
    'Page_Range': [86],
    'EndPart': 'Btm',
    'Positions': {'nan': {'StrPage': None,
      'StrLocation': None,
      'Part': None,
      'EndLocation': 0,
      'EndPage': 86}}},
   '業務課': {'StrPage': 86,
    'StrLocation': 255,
    'StrPart': 'Btm',
    'EndPage': 87,
    'EndLocation': 500,
    'Page_Range': [86, 87],
    'EndPart': 'Btm',
    'Positions': {'nan': {'StrPage': None,
      'StrLocation': None,
      'Part': None,
      'EndLocation': 0,
      'EndPage': 87}}},
   '副収入役室': {'StrPage': 87,
    'StrLocation': 500,
    'EndPage': 87,
    'EndLocation': 245,
    'Page_Range': [87],
    'EndPart': 'Btm',
    'StrPart': 'Btm',
    'Positions': {'nan': {'StrPage': No

# 3. Save Data

In [ ]:
save_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+str(Year)
os.chdir(save_path)
MainDF.to_csv('Proto.csv', index = False, encoding="utf-8")

In [6]:
display(NoImageOffice)
display(IndexError)

,Dept,Office,Posi,Page
0,経理局,会計課,Worker,1
0,経理局,収納課,Worker,2
0,建築部,営繕課,Worker,2
0,戦時生活局,町会課,Worker,2
0,健民局,防疫課,Worker,2
0,健民局,厚生課,Worker,4
0,教育局,学校体育課,Worker,2
0,教育局,日比谷図書館,Worker,2
0,防衛局,防火改修課,Worker,2
0,土木局,道路管理課,Worker,2


,Dept,Office,Posi,Page
0,経理局,会計課,Worker,1
0,経理局,収納課,Worker,2
0,建築部,営繕課,Worker,2
0,戦時生活局,町会課,Worker,2
0,健民局,防疫課,Worker,2
0,健民局,厚生課,Worker,4
0,教育局,学校体育課,Worker,2
0,教育局,日比谷図書館,Worker,2
0,防衛局,防火改修課,Worker,2
0,土木局,道路管理課,Worker,2


# 4. Check Source of error

- Estimated sample with starting position correctly detected.

In [94]:
len(df)*sum(df1['StartPosi']=='True')/len(df1)

0.0

In [95]:
len(df)*sum(df1['StartPosi']=='False')/len(df1)

269.0

- Missing Data 

In [96]:
sum(df1['Posi']=='NoOffice')/len(df1)

0.0

- Unknown Error

In [97]:
sum(df1['Posi']=='Error')/len(df1)

0.0